In [ ]:
import os
import sys
import shutil
import subprocess
import time
from datetime import datetime
from google.colab import files
from IPython.display import display, Markdown

In [ ]:
# 安裝編譯 ffmpeg 所需的相依套件
!sudo apt update
!sudo apt install -y build-essential pkg-config yasm nasm curl xz-utils libmp3lame-dev

In [ ]:
# 將 bash 腳本寫入檔案
script = r'''
set -euo pipefail
export DEBIAN_FRONTEND=noninteractive

cd /content
rm -rf ffmpeg-5.1.9 ffmpeg-5.1.9.tar.xz ffmpeg-5.1.9-build

curl -L --progress-bar -o ffmpeg-5.1.9.tar.xz \
  https://ffmpeg.org/releases/ffmpeg-5.1.9.tar.xz

tar -xf ffmpeg-5.1.9.tar.xz
cd ffmpeg-5.1.9

./configure \
  --prefix=/content/ffmpeg-5.1.9-build \
  --enable-shared \
  --disable-static \
  --disable-doc \
  --disable-debug \
  --enable-gpl \
  --enable-libmp3lame

make -j"$(nproc)"
make install
'''

with open("/content/build_ffmpeg5_mp3.sh", "w") as f:
    f.write(script)

!chmod +x /content/build_ffmpeg5_mp3.sh

In [ ]:
# 執行 bash 腳本，並即時顯示輸出
process = subprocess.Popen(
    "stdbuf -oL -eL bash /content/build_ffmpeg5_mp3.sh",
    shell=True,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1,
)

for line in process.stdout:
    print(line, end="", flush=True)

return_code = process.wait()
print("return code:", return_code)

In [ ]:
# 針對 ffmpeg 5.1.9 的動態連結庫，將其路徑加入到系統的動態連結庫搜尋路徑中
!echo /content/ffmpeg-5.1.9-build/lib | sudo tee /etc/ld.so.conf.d/ffmpeg5.conf
!sudo ldconfig

In [ ]:
# 檢查 ffmpeg 版本
!/content/ffmpeg-5.1.9-build/bin/ffmpeg -version | head -n 1

In [ ]:
# ffmpeg5_bin = "/content/ffmpeg-5.1.9-build/bin"
# ffmpeg5_lib = "/content/ffmpeg-5.1.9-build/lib"

# os.environ["PATH"] = ffmpeg5_bin + ":" + os.environ["PATH"]
# os.environ["LD_LIBRARY_PATH"] = ffmpeg5_lib + ":" + os.environ.get("LD_LIBRARY_PATH", "")

# print("ffmpeg:", shutil.which("ffmpeg"))
# print("LD_LIBRARY_PATH:", os.environ["LD_LIBRARY_PATH"])

os.environ["PATH"] = "/content/ffmpeg-5.1.9-build/bin:" + os.environ["PATH"]
os.environ["LD_LIBRARY_PATH"] = (
    "/content/ffmpeg-5.1.9-build/lib:"
    "/usr/local/lib/python3.12/dist-packages/torch/lib:"
    + os.environ.get("LD_LIBRARY_PATH", "")
)

print(shutil.which("ffmpeg"))

In [ ]:
# 從 GitHub clone heartlib 倉庫，如果已經存在則跳過。
if not os.path.exists("/content/heartlib"):
    !git clone https://github.com/HeartMuLa/heartlib.git /content/heartlib -q

In [ ]:
# 切換到 heartlib 目錄並安裝相關套件
%cd /content/heartlib
!pip install -q -e .

In [ ]:
# 安裝 torchcodec 套件 (版本 0.10.0，對應 CUDA 12.8 和 PyTorch 2.10.0)
!pip install torch==2.10.0 torchvision==0.25.0 torchaudio==2.10.0 --index-url https://download.pytorch.org/whl/cu128
!pip install torchcodec==0.10.0+cu128 --index-url https://download.pytorch.org/whl/cu128

In [ ]:
import sys
import torch
import importlib.metadata as md

print("python:", sys.version)
print("torch:", torch.__version__)
print("torch cuda:", torch.version.cuda)

'''
一般來說會輸出這個訊息:
python: 3.12.13 (main, Mar  4 2026, 09:23:07) [GCC 11.4.0]
torch: 2.10.0+cu128
torch cuda: 12.8
torchcodec: 0.10.0+cu128
'''

try:
    print("torchcodec:", md.version("torchcodec"))
except Exception as e:
    print("torchcodec not found:", e)

In [ ]:
# 切換到 heartlib 目錄
%cd /content/heartlib

# 建立 ckpt 目錄，如果已經存在則不會報錯
os.makedirs('./ckpt', exist_ok=True)

# 從 Hugging Face 下載模型檔案，如果已經存在則跳過下載
if not os.path.exists('./ckpt/HeartMuLa-oss-3B'):
    !huggingface-cli download --local-dir './ckpt/HeartMuLa-oss-3B' 'benjiaiplayground/HeartMuLa-oss-3B-bf16' --quiet

if not os.path.exists('./ckpt/HeartCodec-oss'):
    !huggingface-cli download --local-dir './ckpt/HeartCodec-oss' 'benjiaiplayground/HeartCodec-oss-bf16' --quiet

if not os.path.exists('./ckpt/HeartMuLaGen'):
    !huggingface-cli download --local-dir './ckpt' 'HeartMuLa/HeartMuLaGen' --quiet

# 移動模型檔案到指定位置
codec_target_file = './ckpt/HeartCodec-oss/model.safetensors'
codec_source_file = './ckpt/HeartCodec-oss/HeartCodec-oss-bf16.safetensors'
if not os.path.exists(codec_target_file):
     shutil.move(codec_source_file, codec_target_file)

# 列出 ckpt 目錄下的檔案以確認下載和移動是否成功
!ls -la ./ckpt/

In [ ]:
# 切換到 /content/heartlib
%cd /content/heartlib

# 直接用變數存放 lyrics / tags
lyrics = """\
[Verse]
你的身邊 總有人陪着
我只能在 角落裡附和
藏起眼神裡 溫熱的顏色
裝作只是 朋友的資格

[Chorus]
多想能 走向你呢
這秘密 快要藏不住了
你的一顰一笑 都是我的副歌
卻只能 遠遠地看著"""
tags = "soft,sad,ballad,longing"

# 設定生成參數
duration = 30
temperature = 1.0
topk = 50
cfg_scale = 1.5

# 記錄開始時間，生成檔案名稱，並建立 assets 目錄
start_time = time.time()
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
output_path = f"./assets/music_{timestamp}.mp3"
os.makedirs("./assets", exist_ok=True)

# 將 lyrics 和 tags 寫入臨時檔案
lyrics_file = f"./assets/lyrics_{timestamp}.txt"
tags_file = f"./assets/tags_{timestamp}.txt"
with open(lyrics_file, "w", encoding="utf-8") as f:
    f.write(lyrics)
with open(tags_file, "w", encoding="utf-8") as f:
    f.write(tags)

# 建立命令列參數，呼叫 run_music_generation.py 生成音樂
cmd = [
    "python", "./examples/run_music_generation.py",
    "--model_path=./ckpt",
    "--version=3B",
    "--lazy_load=true",
    f"--lyrics={lyrics_file}",
    f"--tags={tags_file}",
    f"--max_audio_length_ms={int(duration * 1000)}",
    f"--temperature={temperature}",
    f"--topk={topk}",
    f"--cfg_scale={cfg_scale}",
    f"--save_path={output_path}"
]

# 執行命令並捕捉輸出
result = subprocess.run(cmd, capture_output=True, text=True)

# 清理臨時檔案
try:
    os.remove(lyrics_file)
    os.remove(tags_file)
except:
    pass

# 檢查生成結果，若成功則顯示下載連結，否則顯示錯誤訊息
if result.returncode == 0 and os.path.exists(output_path):
    elapsed = time.time() - start_time
    display(Markdown(f"✅ 生成完成：`{output_path}`  \n耗時：{elapsed/60:.1f} 分鐘"))
    files.download(output_path)
else:
    error_msg = result.stderr if result.stderr else result.stdout
    print("❌ 生成失敗")
    print(error_msg[-2000:])
